# 02 — Feature Engineering
**Dataset:** Fruits 360  
**Tujuan Notebook:** Ekstraksi fitur dari gambar menggunakan HOG dan Color Histogram, lalu bandingkan hasilnya.

## 1. Import Library

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import hog
from skimage import exposure
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = 'data/processed'
print('Libraries loaded!')

## 2. Load Data dari Notebook 01

In [ ]:
X_train_flat = np.load(f'{OUTPUT_DIR}/X_train_flat.npy')
X_test_flat  = np.load(f'{OUTPUT_DIR}/X_test_flat.npy')
y_train      = np.load(f'{OUTPUT_DIR}/y_train.npy')
y_test       = np.load(f'{OUTPUT_DIR}/y_test.npy')
label_names  = pd.read_csv(f'{OUTPUT_DIR}/label_names.csv')['class_name'].tolist()

IMG_SIZE = (64, 64)
# Reshape kembali ke (N, H, W, C)
X_train_img = X_train_flat.reshape(-1, IMG_SIZE[0], IMG_SIZE[1], 3)
X_test_img  = X_test_flat.reshape(-1, IMG_SIZE[0], IMG_SIZE[1], 3)

print(f'X_train_img shape: {X_train_img.shape}')
print(f'X_test_img  shape: {X_test_img.shape}')

## 3. Fitur 1: HOG (Histogram of Oriented Gradients)

In [ ]:
def extract_hog_features(images):
    """
    Ekstrak fitur HOG dari array gambar.
    HOG menangkap informasi struktur dan tepi objek.
    """
    features = []
    for img in images:
        feat, _ = hog(
            img,
            orientations=8,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            channel_axis=-1,
            visualize=True
        )
        features.append(feat)
    return np.array(features)

print('Mengekstrak fitur HOG...')
X_train_hog = extract_hog_features(X_train_img)
X_test_hog  = extract_hog_features(X_test_img)
print(f'HOG train shape: {X_train_hog.shape}')
print(f'HOG test  shape: {X_test_hog.shape}')

In [ ]:
# Visualisasi HOG pada 1 gambar contoh
sample_img = X_train_img[0]
_, hog_image = hog(
    sample_img, orientations=8, pixels_per_cell=(8,8),
    cells_per_block=(2,2), channel_axis=-1, visualize=True
)
hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(sample_img)
ax1.set_title('Gambar Original')
ax1.axis('off')
ax2.imshow(hog_image_rescaled, cmap='gray')
ax2.set_title('HOG Features')
ax2.axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/hog_visualization.png', dpi=100)
plt.show()
print(f'Kelas gambar ini: {label_names[y_train[0]]}')

## 4. Fitur 2: Color Histogram

In [ ]:
def extract_color_histogram(images, bins=32):
    """
    Ekstrak histogram warna untuk setiap channel RGB.
    Cocok untuk membedakan buah berdasarkan warna.
    """
    features = []
    for img in images:
        hist_r = np.histogram(img[:,:,0], bins=bins, range=(0,1))[0]
        hist_g = np.histogram(img[:,:,1], bins=bins, range=(0,1))[0]
        hist_b = np.histogram(img[:,:,2], bins=bins, range=(0,1))[0]
        # Normalisasi dan gabungkan
        hist = np.concatenate([hist_r, hist_g, hist_b]).astype(float)
        hist /= hist.sum() + 1e-7
        features.append(hist)
    return np.array(features)

print('Mengekstrak Color Histogram...')
X_train_hist = extract_color_histogram(X_train_img)
X_test_hist  = extract_color_histogram(X_test_img)
print(f'Color Hist train shape: {X_train_hist.shape}')
print(f'Color Hist test  shape: {X_test_hist.shape}')

## 5. Gabungkan Fitur HOG + Color Histogram

In [ ]:
# Menggabungkan kedua fitur untuk representasi yang lebih kaya
X_train_combined = np.hstack([X_train_hog, X_train_hist])
X_test_combined  = np.hstack([X_test_hog,  X_test_hist])

print(f'Combined train shape: {X_train_combined.shape}')
print(f'Combined test  shape: {X_test_combined.shape}')

## 6. Reduksi Dimensi dengan PCA

In [ ]:
# PCA untuk mengurangi dimensi dan mempercepat training
pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_combined)
X_test_pca  = pca.transform(X_test_combined)

explained_var = np.sum(pca.explained_variance_ratio_) * 100
print(f'PCA train shape : {X_train_pca.shape}')
print(f'Variance retained: {explained_var:.2f}%')

# Plot explained variance
plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_) * 100, color='steelblue')
plt.xlabel('Jumlah Komponen PCA')
plt.ylabel('Explained Variance (%)')
plt.title('PCA — Cumulative Explained Variance')
plt.axhline(y=explained_var, color='red', linestyle='--', label=f'{explained_var:.1f}% dengan 100 komponen')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pca_variance.png', dpi=100)
plt.show()

## 7. Simpan Fitur

In [ ]:
np.save(f'{OUTPUT_DIR}/X_train_hog.npy',      X_train_hog)
np.save(f'{OUTPUT_DIR}/X_test_hog.npy',       X_test_hog)
np.save(f'{OUTPUT_DIR}/X_train_hist.npy',     X_train_hist)
np.save(f'{OUTPUT_DIR}/X_test_hist.npy',      X_test_hist)
np.save(f'{OUTPUT_DIR}/X_train_pca.npy',      X_train_pca)
np.save(f'{OUTPUT_DIR}/X_test_pca.npy',       X_test_pca)

print('Semua fitur berhasil disimpan!')

## 8. Ringkasan
| Metode Fitur | Dimensi | Keterangan |
|---|---|---|
| Pixel Flatten | 64×64×3 = 12.288 | Raw pixel, baseline |
| HOG | ~1.152 | Struktur & tepi gambar |
| Color Histogram | 96 | Distribusi warna RGB |
| HOG + Histogram | ~1.248 | Kombinasi keduanya |
| PCA (100 komp.) | 100 | Reduksi dimensi |

Lanjut ke notebook `03_model_training.ipynb`